# 01: SmolVLAの推論

観測をSmolVLAに渡し、返ってきた行動をLift環境で実行します。依存ライブラリと学習済みモデルは実行時に取得します。

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `01_smolvla_inference` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。

同じフォルダに置くファイル

- `lift.json`
- `lift_tools.py`
- `prepare.py`
- `requirements.txt`
- `smolvla.py`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/01_smolvla_inference"  # @param {type:"string"}
REQUIRED_FILES = [
    "lift.json",
    "lift_tools.py",
    "prepare.py",
    "requirements.txt",
    "smolvla.py",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_01")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## モデルを読み込み、行動を予測する処理

ここでは後のセルで使う処理を用意します。モデルを読み込むのは、実習を実行するセルです。

In [ ]:
MODEL_CODE = r"""
import torch
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla import processor_smolvla  # noqa: F401

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

from smolvla import checkpoint_inputs, observation_batch


def load_model(settings, image_size=256):
    checkpoint, config, vlm_repo = checkpoint_inputs(settings, image_size)
    policy = SmolVLAPolicy.from_pretrained(checkpoint, config=config, strict=True).eval()
    # 正規化・トークン化・行動の逆正規化はチェックポイント付属のものを使います。
    pre, post = make_pre_post_processors(
        config,
        pretrained_path=checkpoint,
        preprocessor_overrides={
            "device_processor": {"device": config.device},
            "tokenizer_processor": {"tokenizer_name": config.vlm_model_name},
        },
        postprocessor_overrides={"device_processor": {"device": "cpu"}},
    )
    return policy, pre, post, vlm_repo


def predict_action(policy, pre, post, obs, instruction, settings):
    batch = observation_batch(obs, instruction, settings)
    with torch.inference_mode():
        action = post(policy.select_action(pre(batch)))
    return action.squeeze(0).detach().cpu().numpy()

"""

## モデルへの入力を確認する

前の回の固定行動を、SmolVLAが予測した行動に置き換えます。モデルに渡すのは、2視点の画像、アームの状態、立方体を持ち上げる指示です。

In [ ]:
settings = json.loads((PROJECT / "lift.json").read_text())
policy_config = settings["policy"]
print("モデル:", policy_config["path"])
print("固定revision:", policy_config["revision"])
print("カメラ:", policy_config["camera_map"])
print("指示:", settings["instruction"])

## SmolVLAを読み込み、行動を実行する

In [ ]:
result = run_python(
    MODEL_CODE
    + r"""
import json
import os
import tempfile
from pathlib import Path

import imageio.v2 as imageio
import numpy as np

# 描画バックエンドをrobosuiteのimportより先に設定します。
settings = json.loads(Path("lift.json").read_text())
config = settings["env"]
os.environ["MUJOCO_GL"] = config["renderer"]
os.environ["PYOPENGL_PLATFORM"] = config["renderer"]
import robosuite
from robosuite import macros
from lift_tools import lift_controller, state_vector

macros.IMAGE_CONVENTION = "opengl"
torch.manual_seed(settings["seed"])
policy, pre, post, _ = load_model(settings["policy"], config["image_size"])
env = robosuite.make(
    env_name=config["task"],
    robots=config["robot"],
    controller_configs=lift_controller(),
    has_renderer=False,
    has_offscreen_renderer=True,
    use_camera_obs=True,
    use_object_obs=False,
    camera_names=config["cameras"],
    camera_heights=config["image_size"],
    camera_widths=config["image_size"],
    control_freq=config["control_freq"],
    horizon=settings["max_steps"] + config["settle_steps"],
    hard_reset=False,
    reward_shaping=False,
    seed=settings["seed"],
)
try:
    obs = env.reset()
    for _ in range(config["settle_steps"]):
        obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])

    folder = Path(tempfile.mkdtemp(prefix="lift-", dir="outputs")).resolve()
    camera_key = f'{config["cameras"][0]}_image'
    imageio.imwrite(folder / "observation.png", obs[camera_key][::-1])
    steps = []
    with imageio.get_writer(folder / "video.mp4", fps=config["control_freq"]) as video:
        video.append_data(obs[camera_key][::-1])
        for step in range(settings["max_steps"]):
            state = state_vector(obs)
            proposed = predict_action(
                policy, pre, post, obs, settings["instruction"], settings["policy"]
            )
            action = np.clip(proposed, *env.action_spec)
            obs, reward, done, _ = env.step(action)
            video.append_data(obs[camera_key][::-1])
            steps.append(
                {
                    "step": step,
                    "state_before": state.tolist(),
                    "action": action.tolist(),
                    "reward": reward,
                }
            )
            if done or reward > 0:
                break
    report = {
        "camera_shape": list(obs[camera_key].shape),
        "state_order": ["x", "y", "z", "rx", "ry", "rz", "gripper_left", "gripper_right"],
        "action_order": ["dx", "dy", "dz", "drx", "dry", "drz", "gripper"],
        "steps": steps,
    }
    report["model"] = settings["policy"]["path"]
    report["instruction"] = settings["instruction"]
    (folder / "observations.json").write_text(json.dumps(report, indent=2))
    print(f"Results: {folder}")
finally:
    env.close()
"""
)

## モデルと動作を確認する

In [ ]:
from IPython.display import Image, Video, display

display(Image(filename=str(result / "observation.png")))
display(Video(str(result / "video.mp4"), embed=True))
report = json.loads((result / "observations.json").read_text())
print("カメラ画像の形:", report["camera_shape"])
print("状態の順番:", report["state_order"])
print("行動の順番:", report["action_order"])
print("最初のステップ:", report["steps"][0])
print("モデル:", report["model"])

## 確認課題

モデルに渡した状態・画像・指示と、最初に返った行動を確認してください。

## 結果をDriveに保存する

推論時の動画、実行設定、状態・行動ログを保存します。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = (
    "01_smolvla_inference_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
)
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)